# Notebook 07 — DeepDTA Model

**PharmaLens: An Explainable AI Framework for Drug–Target Interaction Prediction**

---

## Objective

Train a **DeepDTA** model — the base paper architecture for our project.

**DeepDTA** (Öztürk et al., 2018) predicts binding affinity using:
- **Drug branch:** SMILES → character-level embedding → 3 Conv1D layers → GlobalMaxPool
- **Protein branch:** Sequence → character-level embedding → 3 Conv1D layers → GlobalMaxPool
- **Fusion:** Concatenated representations → FC(1024) → FC(512) → score

This is the first deep learning model in our comparison pipeline.

In [ ]:
import os, sys, warnings, pickle, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.evaluation.metrics import compute_regression_metrics
from src.evaluation.experiment_tracker import ExperimentTracker
from src.models.deeptda import DeepDTA
from src.models.deeptda_dataset import DTIDataset, create_encoding_dicts
from src.training.trainer import Trainer

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()

DEVICE = get_device()
tracker = ExperimentTracker()

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print("Setup complete ✓")

## 1. Prepare Data

In [ ]:
# Load clean data and splits
df = pd.read_csv(PROCESSED_DATA_DIR / 'kiba_clean.csv')
train_df = pd.read_csv(SPLITS_DIR / 'random_train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'random_val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'random_test.csv')

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Create encoding dictionaries
DRUG_MAX_LEN = 100
PROTEIN_MAX_LEN = 1000

drug_enc, target_enc = create_encoding_dicts(
    df, drug_max_len=DRUG_MAX_LEN, target_max_len=PROTEIN_MAX_LEN
)
print(f"Encoded {len(drug_enc)} drugs, {len(target_enc)} targets")

In [ ]:
# Create PyTorch Datasets
train_dataset = DTIDataset(
    train_df['drug_id'], train_df['target_id'], train_df['score'],
    drug_enc, target_enc, DRUG_MAX_LEN, PROTEIN_MAX_LEN
)
val_dataset = DTIDataset(
    val_df['drug_id'], val_df['target_id'], val_df['score'],
    drug_enc, target_enc, DRUG_MAX_LEN, PROTEIN_MAX_LEN
)
test_dataset = DTIDataset(
    test_df['drug_id'], test_df['target_id'], test_df['score'],
    drug_enc, target_enc, DRUG_MAX_LEN, PROTEIN_MAX_LEN
)

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val: {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Test: {len(test_dataset)} samples, {len(test_loader)} batches")

## 2. Initialize Model

In [ ]:
model = DeepDTA(
    drug_vocab_size=63,
    protein_vocab_size=21,
    drug_embed_dim=128,
    protein_embed_dim=128,
    num_filters=32,
    drug_filter_sizes=[4, 6, 8],
    protein_filter_sizes=[4, 8, 12],
    fc_dims=[1024, 512],
    dropout=0.2,
    drug_max_len=DRUG_MAX_LEN,
    protein_max_len=PROTEIN_MAX_LEN,
)

print(f"Model: DeepDTA")
print(f"Parameters: {model.count_parameters():,}")
print(f"Device: {DEVICE}")
print(model)

## 3. Train

In [ ]:
# Optimizer and scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
)

# Create trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=nn.MSELoss(),
    device=DEVICE,
    scheduler=scheduler,
    early_stopping_patience=30,
    checkpoint_dir=DEEPTDA_MODELS_DIR,
    model_name='DeepDTA',
)

# Train
EPOCHS = 200
history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)
print(f"\nTotal training time: {trainer.total_training_time:.1f}s")

In [ ]:
# Plot training curves
trainer.plot_training_curves(save_path=FIGURES_DIR / 'deeptda_training_curves.png')
trainer.save_history(EXPERIMENTS_DIR / 'logs' / 'deeptda_history.csv')

## 4. Evaluation

In [ ]:
# Evaluate on all splits
_, y_train_pred, y_train_true = trainer.validate(train_loader)
_, y_val_pred, y_val_true = trainer.validate(val_loader)
_, y_test_pred, y_test_true = trainer.validate(test_loader)

train_metrics = compute_regression_metrics(y_train_true, y_train_pred)
val_metrics = compute_regression_metrics(y_val_true, y_val_pred)
test_metrics = compute_regression_metrics(y_test_true, y_test_pred)

print(f"\n{'='*60}")
print(f"  DeepDTA RESULTS")
print(f"{'='*60}")
for split_name, metrics in [('Train', train_metrics), ('Val', val_metrics), ('Test', test_metrics)]:
    print(f"  {split_name:6s} — RMSE: {metrics['rmse']:.4f}  MAE: {metrics['mae']:.4f}  "
          f"Pearson: {metrics['pearson']:.4f}  Spearman: {metrics['spearman']:.4f}  "
          f"R²: {metrics['r2']:.4f}  CI: {metrics['ci']:.4f}")
print(f"{'='*60}")

# Log experiment
tracker.log_experiment(
    model='DeepDTA', dataset='KIBA',
    representation='smiles_encoding+seq_encoding',
    split='random', seed=42,
    hyperparameters=model.config,
    metrics=test_metrics,
    training_time=trainer.total_training_time,
)

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_true, y_test_pred, alpha=0.15, s=5, color=MODEL_COLORS['DeepDTA'])
lims = [min(y_test_true.min(), y_test_pred.min()), max(y_test_true.max(), y_test_pred.max())]
axes[0].plot(lims, lims, 'k--', alpha=0.5)
axes[0].set_xlabel('Actual KIBA Score')
axes[0].set_ylabel('Predicted KIBA Score')
axes[0].set_title(f'DeepDTA: Actual vs Predicted\nRMSE={test_metrics["rmse"]:.4f}, r={test_metrics["pearson"]:.4f}', fontweight='bold')

residuals = y_test_true - y_test_pred
axes[1].hist(residuals, bins=60, color=MODEL_COLORS['DeepDTA'], alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('DeepDTA: Residual Distribution', fontweight='bold')
axes[1].axvline(0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'deeptda_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## Observations

1. DeepDTA learns directly from raw SMILES and protein sequences without explicit feature engineering, unlike the baseline/XGBoost models.

2. Training on CPU is feasible but slower — the training time is logged for comparison.

3. The gap between train and validation loss indicates the degree of overfitting. Early stopping prevents excessive overfitting.

4. **Comparison with XGBoost:** Both models should be compared on the same test split using the same metrics. XGBoost uses handcrafted features while DeepDTA learns features automatically.

## Conclusion

DeepDTA has been trained, evaluated, and logged. Results will be compared with other models in Notebook 09.

**Next:** [Notebook 08 — GNN Model](./08_gnn_model.ipynb)